In [1]:
# Libraries required to install before executing this model

# pip install numpy
# pip install pandas
# pip install torch
# pip install PyPDF
# pip install faiss-gpu
# pip install tranformers
# pip install sentence-transformers
# pip install sentencepiece

In [2]:
import numpy as np

In [3]:
import faiss

In [4]:
from sentence_transformers import SentenceTransformer

C:\Users\LNMIIT\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5078.98it/s]


In [6]:
from transformers import pipeline

In [7]:
import re

In [8]:
import json

In [9]:
from huggingface_hub import login
login()

In [10]:
import time

In [11]:
#index = faiss.read_index("Aryu1_index.faiss")

In [12]:
query = input("Enter your query: ")

Enter your query:  Suggest Ayurvedic medicine for people with diabetes.


In [13]:
start_time = time.perf_counter()

In [14]:
query="".join(query)

In [15]:
query

'Suggest Ayurvedic medicine for people with diabetes.'

In [16]:
embeddings = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True,  # Useful for cosine similarity
    show_progress_bar=True
)

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.81it/s]


In [17]:
embeddings = embeddings.astype("float32")

In [18]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

In [19]:
print("Embedding shape:", embeddings.shape)
print("First embedding length:", len(embeddings[0]))

Embedding shape: (1, 384)
First embedding length: 384


In [20]:
object = faiss.read_index("objective_index.faiss")

In [21]:
scores, indices = object.search(embeddings, object.ntotal)

In [22]:
choice=indices[0][0]

In [23]:
choice

np.int64(2)

In [24]:
print(f"Scores:{scores} and indices:{indices}")

Scores:[[0.62456906 0.5825952  0.57791126 0.48700985 0.4405708 ]] and indices:[[2 3 1 0 4]]


In [25]:
def open_file(choice):
    match choice:
        case 0:
            index = faiss.read_index("Aryu1_index.faiss")
            with open("Aryu1.json", "r") as file:
                simple_chunks = json.load(file)

        case 1:
            index = faiss.read_index("Aryu2_index.faiss")
            with open("Aryu2.json", "r") as file:
                simple_chunks = json.load(file)

        case 2:
            index = faiss.read_index("Aryu3_index.faiss")
            with open("Aryu3.json", "r") as file:
                simple_chunks = json.load(file)

        case 3:
            index = faiss.read_index("CharakaS_index.faiss")
            with open("CharakaS.json", "r") as file:
                simple_chunks = json.load(file)

        case 4:
            index = faiss.read_index("SushrutaS_index.faiss")
            with open("SushrutaS.json", "r") as file:
                simple_chunks = json.load(file)

        case _:
            print("Invalid Reading")
            return None, None

    print("Reading Complete")
    return index, simple_chunks


In [26]:
index, simple_chunks=open_file(choice)

Reading Complete


In [27]:
#index = faiss.read_index("Aryu2_index.faiss")

In [28]:
index.ntotal

6360

In [29]:
# with open("Aryu2.json", "r") as file:
#     simple_chunks = json.load(file)

In [30]:
len(simple_chunks)

6360

In [31]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

k = 5
scores, indices = index.search(query_embedding, k)
print("Top Results:")
for score, idx in zip(scores[0], indices[0]):
    print(f"Score: {score:.4f} and indices: {indices[0]}")
    print("Text:", simple_chunks[idx])
    print()

Top Results:
Score: 0.7836 and indices: [1163 1389 1407 1228 1453]
Text: ...................................................  123
 
1366_C07.fm  Page 101  Wednesday, August 27, 2003  9:15 AM
© 2004 by CRC Press LLC 
102
 
Scientiﬁc Basis for Ayurvedic Therapies
 
7.1 Introduction
 
Diabetes mellitus (DM) is described in Ayurveda as 
 
madhumeha
 
 
 
kshaudrameha
 
,
 
 

Score: 0.7589 and indices: [1163 1389 1407 1228 1453]
Text: nsiderable information was available about some of the herbs, namely, gymnema,
momordica, trigonella, coccinia, pterocarpus, and ﬁcus, whereas other herbs have not been
studied as extensively. This makes Ayurveda a considerable choice as an antidiabetic
therapy that is safe, effective, and economic.

Score: 0.7511 and indices: [1163 1389 1407 1228 1453]
Text: ular
complications in patients with adult-onset diabetes. II. Motality results, Diabetes, 19, 789, 1970.
15. Bone, K., Gymnema sylvester, in Clinical Applications of Ayurvedic and Chinese Herbs: Monograp

In [32]:
retrieved_chunks = [simple_chunks[i] for i in indices[0]]

In [33]:
print("Retrieved Context:\n")
for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"{i}. {chunk}\n")

Retrieved Context:

1. ...................................................  123
 
1366_C07.fm  Page 101  Wednesday, August 27, 2003  9:15 AM
© 2004 by CRC Press LLC 
102
 
Scientiﬁc Basis for Ayurvedic Therapies
 
7.1 Introduction
 
Diabetes mellitus (DM) is described in Ayurveda as 
 
madhumeha
 
 
 
kshaudrameha
 
,
 
 

2. nsiderable information was available about some of the herbs, namely, gymnema,
momordica, trigonella, coccinia, pterocarpus, and ﬁcus, whereas other herbs have not been
studied as extensively. This makes Ayurveda a considerable choice as an antidiabetic
therapy that is safe, effective, and economic.

3. ular
complications in patients with adult-onset diabetes. II. Motality results, Diabetes, 19, 789, 1970.
15. Bone, K., Gymnema sylvester, in Clinical Applications of Ayurvedic and Chinese Herbs: Monographs
for the Western Herbal Practitioner,  Bone, K., Ed., Phytotherapy Press, Warwick Queensland,
A

4. s of Ayurvedic therapy as discussed in Chapter 2 ar e also app

In [34]:
context = "\n\n".join(retrieved_chunks)

In [35]:
augmented_prompt = f"""
You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
{context}

Question:
{query}

Answer:
"""

In [36]:
augmented_prompt

"\nYou are a helpful assistant.\n\nUse the following context to answer the user's question.\nIf the answer is not contained in the context, say you don't know.\n\nContext:\n...................................................  123\n \n1366_C07.fm  Page 101  Wednesday, August 27, 2003  9:15 AM\n© 2004 by CRC Press LLC \n102\n \nScientiﬁc Basis for Ayurvedic Therapies\n \n7.1 Introduction\n \nDiabetes mellitus (DM) is described in Ayurveda as \n \nmadhumeha\n \n \n \nkshaudrameha\n \n,\n \n \n\nnsiderable information was available about some of the herbs, namely, gymnema,\nmomordica, trigonella, coccinia, pterocarpus, and ﬁcus, whereas other herbs have not been\nstudied as extensively. This makes Ayurveda a considerable choice as an antidiabetic\ntherapy that is safe, effective, and economic.\n\nular\ncomplications in patients with adult-onset diabetes. II. Motality results, Diabetes, 19, 789, 1970.\n15. Bone, K., Gymnema sylvester, in Clinical Applications of Ayurvedic and Chinese Herbs:

In [37]:
generator = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it"
)

response = generator(
    augmented_prompt,
    max_new_tokens=800,
    do_sample=False,
    eos_token_id=generator.tokenizer.eos_token_id,
)
text = response[0]["generated_text"]

sentences = re.findall(r'.*?[.!?](?=\s|$)', text, flags=re.DOTALL)
text = "".join(sentences)

print(text)
#print(response[0]["generated_text"])

Loading weights: 100%|██████████| 288/288 [00:00<00:00, 4059.41it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=800) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_cor


You are a helpful assistant.

Use the following context to answer the user's question.
If the answer is not contained in the context, say you don't know.

Context:
...................................................  123
 
1366_C07.fm  Page 101  Wednesday, August 27, 2003  9:15 AM
© 2004 by CRC Press LLC 
102
 
Scientiﬁc Basis for Ayurvedic Therapies
 
7.1 Introduction
 
Diabetes mellitus (DM) is described in Ayurveda as 
 
madhumeha
 
 
 
kshaudrameha
 
,
 
 

nsiderable information was available about some of the herbs, namely, gymnema,
momordica, trigonella, coccinia, pterocarpus, and ﬁcus, whereas other herbs have not been
studied as extensively. This makes Ayurveda a considerable choice as an antidiabetic
therapy that is safe, effective, and economic.

ular
complications in patients with adult-onset diabetes. II. Motality results, Diabetes, 19, 789, 1970.
15. Bone, K., Gymnema sylvester, in Clinical Applications of Ayurvedic and Chinese Herbs: Monographs
for the Western Herbal Pr

In [38]:
end_time = time.perf_counter()

In [39]:
execution_time = end_time - start_time
print(f"Execution time: {execution_time} seconds")

Execution time: 114.81331839998893 seconds
